# TNBC Visium data preparation

In [ ]:
# Load libraries
import scanpy as sc
import anndata as ad
import os
import numpy as np
import sys
import spottedpy as sp
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import squidpy as sq
from tqdm import tqdm

In [45]:
# Read in anndata visium object
path="/Volumes/Lola/ONEDRIVE/Projects/AURIA/03.SpottedpyPublic/breast_cancer_visium.h5ad"
anndata_breast=sc.read_h5ad(path)

In [46]:
# Subset the AnnData object for batches that correspond to the TNBC samples  "CID4465"= 6  ;"CID44971"= 9
batches_to_subset = ['6', '9']  # List of batches you want to subset
TNBC_adata = anndata_breast[anndata_breast.obs['batch'].isin(batches_to_subset)]

print(TNBC_adata.obs.head())


                                array_row  array_col  tumour_cells       EMT  \
AGCTTATAGAGACCTG-1_CID4465-6-6          5         47           1.0  0.094508   
AACTAGCGTATCGCAC-1_CID4465-6-6          4         48           1.0  0.140841   
AACTTTAGCTGCTGAG-1_CID4465-6-6          5         49           1.0  0.155443   
CCCAAGACAGAGTATG-1_CID4465-6-6          4         50           1.0  0.051341   
GGCATCAACGAGCACG-1_CID4465-6-6          5         51           1.0  0.148433   

                                     EPI batch  
AGCTTATAGAGACCTG-1_CID4465-6-6  0.117825     6  
AACTAGCGTATCGCAC-1_CID4465-6-6  0.163470     6  
AACTTTAGCTGCTGAG-1_CID4465-6-6  0.079570     6  
CCCAAGACAGAGTATG-1_CID4465-6-6  0.346057     6  
GGCATCAACGAGCACG-1_CID4465-6-6  0.168688     6  


In [47]:
# Get the cell barcodes for TNBC samples
TNBC_samples = TNBC_adata.obs.index

# Subset the deconvolution results TNBC samples
TNBC_deconvolution = TNBC_adata.obsm['deconvolution_results']
print(TNBC_deconvolution)

                                 B cells Memory  B cells Naive  \
AGCTTATAGAGACCTG-1_CID4465-6-6         0.009680       0.006631   
AACTAGCGTATCGCAC-1_CID4465-6-6         0.059446       0.037291   
AACTTTAGCTGCTGAG-1_CID4465-6-6         0.008187       0.005213   
CCCAAGACAGAGTATG-1_CID4465-6-6         0.012913       0.005039   
GGCATCAACGAGCACG-1_CID4465-6-6         0.014081       0.011512   
...                                         ...            ...   
TGCAGTGGTAGGGAAC-1_CID44971-9-9        0.099814       0.048998   
AGCGAGACGTGAAGGC-1_CID44971-9-9        0.147610       0.056138   
CAGTGTTAATCTCTCA-1_CID44971-9-9        0.080135       0.018698   
GATCGCTGTGGTGCGT-1_CID44971-9-9        0.279668       0.159734   
CTCTGCAGGCATTCTT-1_CID44971-9-9        0.131290       0.054497   

                                 CAFs MSC iCAF-like s1  CAFs MSC iCAF-like s2  \
AGCTTATAGAGACCTG-1_CID4465-6-6                0.053007               0.002948   
AACTAGCGTATCGCAC-1_CID4465-6-6               

In [48]:
# Remove columns with 'nan' or unwanted columns
TNBC_deconvolution = TNBC_deconvolution.loc[:, TNBC_deconvolution.columns != 'nan']


In [49]:
# Merge the deconvolution results with TNBC_adata.obs on the index (cell names)
TNBC_adata.obs = TNBC_adata.obs.merge(TNBC_deconvolution, left_index=True, right_index=True, how='left')

# Check the first few rows of the merged data 
print(TNBC_adata.obs.head())

                                array_row  array_col  tumour_cells       EMT  \
AGCTTATAGAGACCTG-1_CID4465-6-6          5         47           1.0  0.094508   
AACTAGCGTATCGCAC-1_CID4465-6-6          4         48           1.0  0.140841   
AACTTTAGCTGCTGAG-1_CID4465-6-6          5         49           1.0  0.155443   
CCCAAGACAGAGTATG-1_CID4465-6-6          4         50           1.0  0.051341   
GGCATCAACGAGCACG-1_CID4465-6-6          5         51           1.0  0.148433   

                                     EPI batch  B cells Memory  B cells Naive  \
AGCTTATAGAGACCTG-1_CID4465-6-6  0.117825     6        0.009680       0.006631   
AACTAGCGTATCGCAC-1_CID4465-6-6  0.163470     6        0.059446       0.037291   
AACTTTAGCTGCTGAG-1_CID4465-6-6  0.079570     6        0.008187       0.005213   
CCCAAGACAGAGTATG-1_CID4465-6-6  0.346057     6        0.012913       0.005039   
GGCATCAACGAGCACG-1_CID4465-6-6  0.168688     6        0.014081       0.011512   

                                

In [50]:
# Read the csv with the signatures for the different TNBC samples (oxsstress an GCLC_VIM, load the updated version)
signatures_TNBC = pd.read_csv('/Users/spalominoe/Python_jobs/01_scripts/figures/signatures_TNBC_spatial.csv')

# Display the first few rows of the dataframe
print(signatures_TNBC.head())

# Display the dimensions of the dataframe
print(signatures_TNBC.shape)

   Unnamed: 0  X                           coord  oxstress  GCLCVIM_tumor
0           1  1  AGCTTATAGAGACCTG-1_CID4465-6-6  0.122504       0.181922
1           2  2  AACTAGCGTATCGCAC-1_CID4465-6-6  0.155302       0.200289
2           3  3  AACTTTAGCTGCTGAG-1_CID4465-6-6  0.117562       0.184223
3           4  4  CCCAAGACAGAGTATG-1_CID4465-6-6  0.142221       0.169613
4           5  5  GGCATCAACGAGCACG-1_CID4465-6-6  0.142492       0.182334
(2352, 5)


In [51]:
# Step 1: Ensure 'coord' is a pandas Index or Series
# Convert 'coord' to a pandas Index if it isn't already
coord_index = pd.Index(signatures_TNBC['coord'])

# Step 2: Filter the AnnData object based on matching indices
matching_indices = TNBC_adata.obs.index.isin(coord_index)

# Filter the AnnData object based on matching indices in obs
filtered_adata = TNBC_adata[matching_indices, :].copy()

# Step 3: Reorder the filtered AnnData object by the order of 'coord' in signatures_CID44971
# Sort the 'coord_index' and use that order to reorder the obs of the AnnData object
sorted_order = coord_index[coord_index.isin(filtered_adata.obs.index)].argsort()

# Step 4: Apply this sorted order to the filtered_adata
filtered_adata = filtered_adata[filtered_adata.obs.index[sorted_order], :]

# Optionally, check the shape of the resulting AnnData object
print(filtered_adata)


View of AnnData object with n_obs × n_vars = 2352 × 14664
    obs: 'array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch', 'B cells Memory', 'B cells Naive', 'CAFs MSC iCAF-like s1', 'CAFs MSC iCAF-like s2', 'CAFs Transitioning s3', 'CAFs myCAF like s4', 'CAFs myCAF like s5', 'Cancer Basal SC_0', 'Cancer Basal SC_1', 'Cancer Basal SC_2', 'Cancer Cycling_0', 'Cancer Cycling_1', 'Cancer Cycling_2', 'Cancer Her2 SC_0', 'Cancer Her2 SC_1', 'Cancer Her2 SC_2', 'Cancer LumA SC_0', 'Cancer LumA SC_1', 'Cancer LumA SC_2', 'Cancer LumB SC_0', 'Cancer LumB SC_1', 'Cancer LumB SC_2', 'Cycling PVL', 'Cycling_Myeloid', 'Endothelial ACKR1', 'Endothelial CXCL12', 'Endothelial Lymphatic LYVE1', 'Endothelial RGS5', 'Luminal Progenitors_0', 'Luminal Progenitors_1', 'Luminal Progenitors_2', 'Mature Luminal_0', 'Mature Luminal_1', 'Mature Luminal_2', 'Myeloid_c0_DC_LAMP3', 'Myeloid_c10_Macrophage_1_EGR1', 'Myeloid_c11_cDC2_CD1C', 'Myeloid_c12_Monocyte_1_IL1B', 'Myeloid_c1_LAM1_FABP5', 'Myeloid_c2

In [52]:
signatures_TNBC.set_index('coord', inplace=True)

In [53]:
# Reindex to match filtered_adata.obs
filtered_adata.obs['oxstress'] = signatures_TNBC.loc[filtered_adata.obs.index, 'oxstress']
print(filtered_adata.obs['oxstress'])


filtered_adata.obs['GCLCVIM_tumor'] = signatures_TNBC.loc[filtered_adata.obs.index, 'GCLCVIM_tumor']
print(filtered_adata.obs['GCLCVIM_tumor'])

AAACAAGTATCTCCCA-1_CID4465-6-6     0.181864
AAACAATCTACTAGCA-1_CID44971-9-9    0.143019
AAACACCAATAACTGC-1_CID44971-9-9    0.155864
AAACAGAGCGACTCCT-1_CID44971-9-9    0.194430
AAACATTTCCCGGATT-1_CID4465-6-6     0.116097
                                     ...   
TTGTTGTGTGTCAAGA-1_CID44971-9-9    0.186422
TTGTTTCACATCCAGG-1_CID44971-9-9    0.151535
TTGTTTCATTAGTCTA-1_CID44971-9-9    0.174105
TTGTTTGTGTAAATTC-1_CID4465-6-6     0.122554
TTGTTTGTGTAAATTC-1_CID44971-9-9    0.179756
Name: oxstress, Length: 2352, dtype: float64
AAACAAGTATCTCCCA-1_CID4465-6-6     0.200980
AAACAATCTACTAGCA-1_CID44971-9-9    0.139876
AAACACCAATAACTGC-1_CID44971-9-9    0.191770
AAACAGAGCGACTCCT-1_CID44971-9-9    0.200621
AAACATTTCCCGGATT-1_CID4465-6-6     0.148931
                                     ...   
TTGTTGTGTGTCAAGA-1_CID44971-9-9    0.186734
TTGTTTCACATCCAGG-1_CID44971-9-9    0.165856
TTGTTTCATTAGTCTA-1_CID44971-9-9    0.201010
TTGTTTGTGTAAATTC-1_CID4465-6-6     0.199684
TTGTTTGTGTAAATTC-1_CID44971-9-9

In [54]:
# Replace spaces with underscores in filtered_adata.obs column names
filtered_adata.obs.columns = [col.replace(' ', '_') for col in filtered_adata.obs.columns]

# Verify the changes
print(filtered_adata.obs.columns)

Index(['array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch',
       'B_cells_Memory', 'B_cells_Naive', 'CAFs_MSC_iCAF-like_s1',
       'CAFs_MSC_iCAF-like_s2', 'CAFs_Transitioning_s3', 'CAFs_myCAF_like_s4',
       'CAFs_myCAF_like_s5', 'Cancer_Basal_SC_0', 'Cancer_Basal_SC_1',
       'Cancer_Basal_SC_2', 'Cancer_Cycling_0', 'Cancer_Cycling_1',
       'Cancer_Cycling_2', 'Cancer_Her2_SC_0', 'Cancer_Her2_SC_1',
       'Cancer_Her2_SC_2', 'Cancer_LumA_SC_0', 'Cancer_LumA_SC_1',
       'Cancer_LumA_SC_2', 'Cancer_LumB_SC_0', 'Cancer_LumB_SC_1',
       'Cancer_LumB_SC_2', 'Cycling_PVL', 'Cycling_Myeloid',
       'Endothelial_ACKR1', 'Endothelial_CXCL12',
       'Endothelial_Lymphatic_LYVE1', 'Endothelial_RGS5',
       'Luminal_Progenitors_0', 'Luminal_Progenitors_1',
       'Luminal_Progenitors_2', 'Mature_Luminal_0', 'Mature_Luminal_1',
       'Mature_Luminal_2', 'Myeloid_c0_DC_LAMP3',
       'Myeloid_c10_Macrophage_1_EGR1', 'Myeloid_c11_cDC2_CD1C',
       'Myeloid_c12_Monocyte_

In [55]:
# Extract the deconvolution_results matrix for normalization 
deconv_results = filtered_adata.obsm['deconvolution_results']
print(deconv_results)

# Normalize each row to sum to 1
normalized_deconv_results = deconv_results.div(deconv_results.sum(axis=1), axis=0)
print(normalized_deconv_results)

row_sums =normalized_deconv_results.sum(axis=1)
print(row_sums)


                                 B cells Memory  B cells Naive  \
AAACAAGTATCTCCCA-1_CID4465-6-6         1.560966       0.807271   
AAACAATCTACTAGCA-1_CID44971-9-9        0.012493       0.012971   
AAACACCAATAACTGC-1_CID44971-9-9        0.006877       0.003940   
AAACAGAGCGACTCCT-1_CID44971-9-9        0.011761       0.005196   
AAACATTTCCCGGATT-1_CID4465-6-6         0.886215       0.463183   
...                                         ...            ...   
TTGTTGTGTGTCAAGA-1_CID44971-9-9        0.000792       0.000832   
TTGTTTCACATCCAGG-1_CID44971-9-9        0.633509       0.378838   
TTGTTTCATTAGTCTA-1_CID44971-9-9        0.091400       0.050584   
TTGTTTGTGTAAATTC-1_CID4465-6-6         0.008835       0.005342   
TTGTTTGTGTAAATTC-1_CID44971-9-9        0.029801       0.015500   

                                 CAFs MSC iCAF-like s1  CAFs MSC iCAF-like s2  \
AAACAAGTATCTCCCA-1_CID4465-6-6                6.009028               0.107440   
AAACAATCTACTAGCA-1_CID44971-9-9              

In [56]:
# Add '_norm' to all column names if deconv_results is a DataFrame
if isinstance(deconv_results, pd.DataFrame):
    normalized_deconv_results.columns = [f"{col}_norm" for col in deconv_results.columns]
    
filtered_adata.obsm['normalized_deconv_results'] = normalized_deconv_results
filtered_adata

AnnData object with n_obs × n_vars = 2352 × 14664
    obs: 'array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch', 'B_cells_Memory', 'B_cells_Naive', 'CAFs_MSC_iCAF-like_s1', 'CAFs_MSC_iCAF-like_s2', 'CAFs_Transitioning_s3', 'CAFs_myCAF_like_s4', 'CAFs_myCAF_like_s5', 'Cancer_Basal_SC_0', 'Cancer_Basal_SC_1', 'Cancer_Basal_SC_2', 'Cancer_Cycling_0', 'Cancer_Cycling_1', 'Cancer_Cycling_2', 'Cancer_Her2_SC_0', 'Cancer_Her2_SC_1', 'Cancer_Her2_SC_2', 'Cancer_LumA_SC_0', 'Cancer_LumA_SC_1', 'Cancer_LumA_SC_2', 'Cancer_LumB_SC_0', 'Cancer_LumB_SC_1', 'Cancer_LumB_SC_2', 'Cycling_PVL', 'Cycling_Myeloid', 'Endothelial_ACKR1', 'Endothelial_CXCL12', 'Endothelial_Lymphatic_LYVE1', 'Endothelial_RGS5', 'Luminal_Progenitors_0', 'Luminal_Progenitors_1', 'Luminal_Progenitors_2', 'Mature_Luminal_0', 'Mature_Luminal_1', 'Mature_Luminal_2', 'Myeloid_c0_DC_LAMP3', 'Myeloid_c10_Macrophage_1_EGR1', 'Myeloid_c11_cDC2_CD1C', 'Myeloid_c12_Monocyte_1_IL1B', 'Myeloid_c1_LAM1_FABP5', 'Myeloid_c2_LAM2_AP

In [57]:
filtered_adata.write("TNBC_norm_signatures_07.h5ad")

In [58]:
filtered_adata

AnnData object with n_obs × n_vars = 2352 × 14664
    obs: 'array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch', 'B_cells_Memory', 'B_cells_Naive', 'CAFs_MSC_iCAF-like_s1', 'CAFs_MSC_iCAF-like_s2', 'CAFs_Transitioning_s3', 'CAFs_myCAF_like_s4', 'CAFs_myCAF_like_s5', 'Cancer_Basal_SC_0', 'Cancer_Basal_SC_1', 'Cancer_Basal_SC_2', 'Cancer_Cycling_0', 'Cancer_Cycling_1', 'Cancer_Cycling_2', 'Cancer_Her2_SC_0', 'Cancer_Her2_SC_1', 'Cancer_Her2_SC_2', 'Cancer_LumA_SC_0', 'Cancer_LumA_SC_1', 'Cancer_LumA_SC_2', 'Cancer_LumB_SC_0', 'Cancer_LumB_SC_1', 'Cancer_LumB_SC_2', 'Cycling_PVL', 'Cycling_Myeloid', 'Endothelial_ACKR1', 'Endothelial_CXCL12', 'Endothelial_Lymphatic_LYVE1', 'Endothelial_RGS5', 'Luminal_Progenitors_0', 'Luminal_Progenitors_1', 'Luminal_Progenitors_2', 'Mature_Luminal_0', 'Mature_Luminal_1', 'Mature_Luminal_2', 'Myeloid_c0_DC_LAMP3', 'Myeloid_c10_Macrophage_1_EGR1', 'Myeloid_c11_cDC2_CD1C', 'Myeloid_c12_Monocyte_1_IL1B', 'Myeloid_c1_LAM1_FABP5', 'Myeloid_c2_LAM2_AP

In [59]:
# Merge the normalized deconvolution results with TNBC_adata.obs on the index (cell names)
filtered_adata.obs = filtered_adata.obs.merge(normalized_deconv_results, left_index=True, right_index=True, how='left')

# Check all .obs names
obs_names = filtered_adata.obs.columns
print(".obs column names:", obs_names)

.obs column names: Index(['array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch',
       'B_cells_Memory', 'B_cells_Naive', 'CAFs_MSC_iCAF-like_s1',
       'CAFs_MSC_iCAF-like_s2',
       ...
       'T_cells_c1_CD4+_IL7R_norm', 'T_cells_c2_CD4+_T-regs_FOXP3_norm',
       'T_cells_c3_CD4+_Tfh_CXCL13_norm', 'T_cells_c4_CD8+_ZFP36_norm',
       'T_cells_c5_CD8+_GZMK_norm', 'T_cells_c6_IFIT1_norm',
       'T_cells_c7_CD8+_IFNG_norm', 'T_cells_c8_CD8+_LAG3_norm',
       'T_cells_c9_NK_cells_AREG_norm', 'nan_norm'],
      dtype='object', length=137)


In [60]:
TNBC_adata = filtered_adata

In [61]:
# Extract the deconvolution_results to merge cell types
deconv_results = TNBC_adata.obsm['deconvolution_results']
print(deconv_results)

                                 B cells Memory  B cells Naive  \
AAACAAGTATCTCCCA-1_CID4465-6-6         1.560966       0.807271   
AAACAATCTACTAGCA-1_CID44971-9-9        0.012493       0.012971   
AAACACCAATAACTGC-1_CID44971-9-9        0.006877       0.003940   
AAACAGAGCGACTCCT-1_CID44971-9-9        0.011761       0.005196   
AAACATTTCCCGGATT-1_CID4465-6-6         0.886215       0.463183   
...                                         ...            ...   
TTGTTGTGTGTCAAGA-1_CID44971-9-9        0.000792       0.000832   
TTGTTTCACATCCAGG-1_CID44971-9-9        0.633509       0.378838   
TTGTTTCATTAGTCTA-1_CID44971-9-9        0.091400       0.050584   
TTGTTTGTGTAAATTC-1_CID4465-6-6         0.008835       0.005342   
TTGTTTGTGTAAATTC-1_CID44971-9-9        0.029801       0.015500   

                                 CAFs MSC iCAF-like s1  CAFs MSC iCAF-like s2  \
AAACAAGTATCTCCCA-1_CID4465-6-6                6.009028               0.107440   
AAACAATCTACTAGCA-1_CID44971-9-9              

In [62]:
# Sum all columns containing the desired cell type eg."B cells"
deconv_results["B_cells_total"] = deconv_results.filter(like="B cells").sum(axis=1)
deconv_results["CAFs_total"] = deconv_results.filter(like="CAFs").sum(axis=1)
deconv_results["Cancer_total"] = deconv_results.filter(like="Cancer").sum(axis=1)
deconv_results["Endothelial_total"] = deconv_results.filter(like="Endothelial").sum(axis=1)
deconv_results["Luminal_total"] = deconv_results.filter(like="Luminal").sum(axis=1)
deconv_results["Myeloid_total"] = deconv_results.filter(like="Myeloid").sum(axis=1)
deconv_results["Myoepithelia_total"] = deconv_results.filter(like="Myoepithelial").sum(axis=1)
deconv_results["PVL_total"] = deconv_results.filter(like="PVL").sum(axis=1)
deconv_results["TCD4_total"] = deconv_results.filter(like="CD4+").sum(axis=1)
deconv_results["TCD8_total"] = deconv_results.filter(like="CD8+").sum(axis=1)
deconv_results["NK_total"] = deconv_results.filter(like="NK").sum(axis=1)
deconv_results["Plasmablasts_total"] = deconv_results.filter(like="Plasmablasts").sum(axis=1)


In [63]:
# Keep index and only the _total columns
# Select only columns that end with '_total'
total_columns = deconv_results.filter(regex="_total$")

# View result
print(total_columns.head())


                                 B_cells_total  CAFs_total  Cancer_total  \
AAACAAGTATCTCCCA-1_CID4465-6-6        2.368237    6.807418      0.112292   
AAACAATCTACTAGCA-1_CID44971-9-9       0.025464    2.909282      0.105171   
AAACACCAATAACTGC-1_CID44971-9-9       0.010817    0.018237      1.554334   
AAACAGAGCGACTCCT-1_CID44971-9-9       0.016956    0.009710      2.263486   
AAACATTTCCCGGATT-1_CID4465-6-6        1.349398    4.903491      0.314650   

                                 Endothelial_total  Luminal_total  \
AAACAAGTATCTCCCA-1_CID4465-6-6            0.995030       0.067387   
AAACAATCTACTAGCA-1_CID44971-9-9           0.138928       1.675480   
AAACACCAATAACTGC-1_CID44971-9-9           0.001365       0.029285   
AAACAGAGCGACTCCT-1_CID44971-9-9           0.017970       0.005089   
AAACATTTCCCGGATT-1_CID4465-6-6            1.330583       0.035175   

                                 Myeloid_total  Myoepithelia_total  PVL_total  \
AAACAAGTATCTCCCA-1_CID4465-6-6        2.867614 

In [64]:
# Normalize each row to sum to 1
normalized_subset = total_columns.div(total_columns.sum(axis=1), axis=0)
print(normalized_subset)

row_sums =normalized_subset.sum(axis=1)
print(row_sums)

                                 B_cells_total  CAFs_total  Cancer_total  \
AAACAAGTATCTCCCA-1_CID4465-6-6        0.120735    0.347049      0.005725   
AAACAATCTACTAGCA-1_CID44971-9-9       0.003801    0.434293      0.015700   
AAACACCAATAACTGC-1_CID44971-9-9       0.006080    0.010251      0.873688   
AAACAGAGCGACTCCT-1_CID44971-9-9       0.006845    0.003920      0.913707   
AAACATTTCCCGGATT-1_CID4465-6-6        0.076609    0.278385      0.017864   
...                                        ...         ...           ...   
TTGTTGTGTGTCAAGA-1_CID44971-9-9       0.000821    0.005657      0.974378   
TTGTTTCACATCCAGG-1_CID44971-9-9       0.084904    0.208919      0.018575   
TTGTTTCATTAGTCTA-1_CID44971-9-9       0.016452    0.142796      0.055047   
TTGTTTGTGTAAATTC-1_CID4465-6-6        0.002397    0.142113      0.252034   
TTGTTTGTGTAAATTC-1_CID44971-9-9       0.017810    0.230622      0.286156   

                                 Endothelial_total  Luminal_total  \
AAACAAGTATCTCCCA-1

In [65]:
TNBC_adata.obsm['normalized_subset'] = normalized_subset
TNBC_adata

AnnData object with n_obs × n_vars = 2352 × 14664
    obs: 'array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch', 'B_cells_Memory', 'B_cells_Naive', 'CAFs_MSC_iCAF-like_s1', 'CAFs_MSC_iCAF-like_s2', 'CAFs_Transitioning_s3', 'CAFs_myCAF_like_s4', 'CAFs_myCAF_like_s5', 'Cancer_Basal_SC_0', 'Cancer_Basal_SC_1', 'Cancer_Basal_SC_2', 'Cancer_Cycling_0', 'Cancer_Cycling_1', 'Cancer_Cycling_2', 'Cancer_Her2_SC_0', 'Cancer_Her2_SC_1', 'Cancer_Her2_SC_2', 'Cancer_LumA_SC_0', 'Cancer_LumA_SC_1', 'Cancer_LumA_SC_2', 'Cancer_LumB_SC_0', 'Cancer_LumB_SC_1', 'Cancer_LumB_SC_2', 'Cycling_PVL', 'Cycling_Myeloid', 'Endothelial_ACKR1', 'Endothelial_CXCL12', 'Endothelial_Lymphatic_LYVE1', 'Endothelial_RGS5', 'Luminal_Progenitors_0', 'Luminal_Progenitors_1', 'Luminal_Progenitors_2', 'Mature_Luminal_0', 'Mature_Luminal_1', 'Mature_Luminal_2', 'Myeloid_c0_DC_LAMP3', 'Myeloid_c10_Macrophage_1_EGR1', 'Myeloid_c11_cDC2_CD1C', 'Myeloid_c12_Monocyte_1_IL1B', 'Myeloid_c1_LAM1_FABP5', 'Myeloid_c2_LAM2_AP

In [66]:
# Merge the normalized deconvolution results with TNBC_adata.obs on the index (cell names)
TNBC_adata.obs = TNBC_adata.obs.merge(normalized_subset, left_index=True, right_index=True, how='left')

# Check all .obs names
obs_names = TNBC_adata.obs.columns
print(".obs column names:", obs_names)

.obs column names: Index(['array_row', 'array_col', 'tumour_cells', 'EMT', 'EPI', 'batch',
       'B_cells_Memory', 'B_cells_Naive', 'CAFs_MSC_iCAF-like_s1',
       'CAFs_MSC_iCAF-like_s2',
       ...
       'Cancer_total', 'Endothelial_total', 'Luminal_total', 'Myeloid_total',
       'Myoepithelia_total', 'PVL_total', 'TCD4_total', 'TCD8_total',
       'NK_total', 'Plasmablasts_total'],
      dtype='object', length=149)


In [ ]:
TNBC_adata.write("TNBC_spatial.h5ad")